# ⚡ Paralelni agentni tijekovi rada s GitHub modelima (Python)

## 📋 Napredni vodič za paralelnu obradu

Ovaj bilježnik prikazuje **uzorke paralelnih tijekova rada** koristeći Microsoft Agent Framework. Naučit ćete kako izgraditi visokoučinkovite, paralelne tijekove rada u kojima više AI agenata istovremeno izvršava zadatke, značajno poboljšavajući propusnost i omogućujući složene višedretvene poslovne procese.

## 🎯 Ciljevi učenja

### 🚀 **Osnove paralelne obrade**
- **Paralelno izvršavanje agenata**: Pokrenite više agenata istovremeno za maksimalnu učinkovitost
- **Orkestracija tijeka rada**: Koordinirajte paralelne operacije uz održavanje konzistentnosti podataka
- **Optimizacija performansi**: Postignite značajno ubrzanje paralelnom obradom
- **Upravljanje resursima**: Efikasno koristite AI modele kroz paralelne operacije

### 🏗️ **Napredni uzorci paralelizma**
- **Fork-Join obrada**: Podijelite rad na više agenata i spojite rezultate
- **Paralelizam u cjevovodu**: Preklapajuće faze izvršavanja za kontinuiranu propusnost
- **Ravnomjerno raspoređivanje opterećenja**: Dijelite rad ravnomjerno na dostupne agenate
- **Točke sinkronizacije**: Koordinirajte paralelne agente u ključnim fazama tijeka rada

### 🏢 **Višestruke aplikacije za poduzeća**
- **Obrada velikog broja dokumenata**: Obrada više dokumenata istovremeno
- **Analiza sadržaja u stvarnom vremenu**: Paralelna analiza dolaznih podatkovnih tokova
- **Optimizacija obrade u serijama**: Maksimizirajte propusnost za velike operacije
- **Višemodalna analiza**: Paralelna obrada različitih vrsta sadržaja (tekst, slike, podaci)

## ⚙️ Preduvjeti i postavljanje

### 📦 **Potrebne ovisnosti**

Instalirajte Agent Framework s podrškom za paralelne tijekove rada:

```bash
pip install agent-framework-core -U
```

### 🔑 **Konfiguracija GitHub modela**

**Postavljanje okoline (.env datoteka):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Razmatranja o paralelnoj obradi:**
- **Ograničenja brzine**: Pratite ograničenja GitHub Models API-ja za paralelne zahtjeve
- **Korištenje resursa**: Uzmite u obzir potrošnju memorije i CPU-a kod više paralelnih agenata
- **Rukovanje pogreškama**: Implementirajte robusno vraćanje pogrešaka za paralelne operacije

### 🏗️ **Arhitektura paralelnog tijeka rada**

```mermaid
graph TD
    A[Početak tijeka rada] --> B[Istovremeno izvođenje]
    B --> C[Skup agenata 1]
    B --> D[Skup agenata 2]
    B --> E[Skup agenata 3]
    C --> F[Agregacija rezultata]
    D --> F
    E --> F
    F --> G[Konačni izlaz]
    
    H[GitHub Models API] --> C
    H --> D
    H --> E
```

**Ključne prednosti:**
- **⚡ Performanse**: Značajno ubrzanje paralelnim izvršavanjem
- **📈 Skalabilnost**: Podrška za povećane opterećenja bez proporcionalnog povećanja vremena
- **🔄 Učinkovitost**: Bolje iskorištavanje dostupnih računalnih resursa
- **🎯 Propusnost**: Obrada većeg opsega posla u istom vremenu

## 🎨 **Uzorci dizajna paralelnih tijekova rada**

### 🔍 **Cjevovod istraživanja i analize**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Tijek obrade podataka**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Cjevovod stvaranja sadržaja**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Višefazna obrada**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Prednosti performansi za poduzeća**

### ⚡ **Optimizacija propusnosti**
- **Paralelno izvršavanje**: Više agenata koji rade istovremeno
- **Iskorištavanje resursa**: Maksimalna učinkovitost dostupnih AI modela
- **Smanjenje vremena**: Značajno smanjenje ukupnog vremena obrade
- **Skalabilna arhitektura**: Jednostavno dodavanje više paralelnih agenata prema potrebi

### 🛡️ **Pouzdanost i otpornost**
- **Otpornost na pogreške**: Neuspjesi pojedinih agenata ne zaustavljaju cijeli tijek rada
- **Izolacija pogrešaka**: Problemi u jednoj paralelnoj grani ne utječu na druge
- **Uljudno degradiranje**: Sustav nastavlja rad čak i uz smanjeni kapacitet agenata
- **Mehanizmi oporavka**: Automatsko ponavljanje i rukovanje pogreškama za neuspješne operacije

### 📊 **Praćenje i uvidi**
- **Praćenje paralelnog izvršavanja**: Nadgledajte napredak svih paralelnih operacija
- **Mjerne performanse**: Mjerite ubrzanje i povećanje učinkovitosti
- **Analitika upotrebe resursa**: Optimizirajte dodjelu paralelnih agenata
- **Identifikacija uskih grla**: Pronađite i riješite ograničenja performansi

Izgradimo visokoučinkovite paralelne AI tijekove rada! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
